# Insurance Pricing — Severity Model

This notebook trains a **claim severity** model: the expected cost per claim
given that a claim has occurred.

Claim severity is modelled with a **Gamma distribution** because:
- Claim costs are strictly positive continuous values
- The distribution is right-skewed, matching typical loss distributions
- The log link ensures predicted severities are always positive
- The Gamma deviance penalises relative errors, which is appropriate when
  claim sizes span several orders of magnitude

## Training set
Severity is estimated only on **policies with at least one claim** (`n_claims > 0`).
At prediction time, severity is scored on the **full portfolio** — the pure
premium is then assembled as: `pure_premium = frequency × severity`.

## Models
| Model | Description |
|-------|-------------|
| **GLM (baseline)** | `GammaRegressor` — classical actuarial log-linear model |
| **XGBoost (challenger)** | Gradient-boosted trees with `reg:gamma` objective + GridSearchCV |

Both models share the same `Pipeline + FeatureUnion` preprocessing as the
frequency model. The same feature treatment rationale applies.

> **Prerequisite:** Run `01_frequency_model.ipynb` first to generate `data/freq_predictions.csv`.

In [ ]:
import os
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import GammaRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_gamma_deviance

from xgboost import XGBRegressor

from transformers import FeatureSelectorValues, ComputeAgeFromDOB, ComputeVehicleAge

## 00. Load Data

In [ ]:
data = pd.read_csv("data/insurance_data.csv")

claimants = data[(data["n_claims"] > 0) & (data["claim_amount"] > 0)].copy()
claimants.reset_index(drop=True, inplace=True)

print(f"Full portfolio : {len(data):,} policies")
print(f"Claimants only : {len(claimants):,} policies ({len(claimants)/len(data):.1%})")
print(f"\nSeverity statistics (claimants):")
sev = claimants["claim_amount"] / claimants["n_claims"]
print(f"  Mean severity : {sev.mean():,.0f}")
print(f"  Median        : {sev.median():,.0f}")
print(f"  p95           : {sev.quantile(0.95):,.0f}")
print(f"  Max           : {sev.max():,.0f}")

In [ ]:
# Severity distribution — right-skewed, motivating the Gamma assumption
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].hist(sev, bins=60, edgecolor="none", alpha=0.8)
axes[0].set_xlabel("Severity (cost per claim)")
axes[0].set_title("Raw severity distribution")

axes[1].hist(np.log(sev), bins=60, edgecolor="none", alpha=0.8, color="orange")
axes[1].set_xlabel("log(Severity)")
axes[1].set_title("Log severity distribution (approximately normal → Gamma appropriate)")

plt.tight_layout()
plt.show()

## 01. Prepare Target and Split

- **Training set**: claimants only (`n_claims > 0`)
- **Target** `y`: average cost per claim = `claim_amount / n_claims`
- **Sample weight** `w`: number of claims (policies with more claims contribute more)
- **Prediction set**: full portfolio (including zero-claim policies)

Using the same `random_state=42` as the frequency notebook ensures the test
split is identical, allowing pure premium assembly in `03_validation.ipynb`.

In [ ]:
# Split on the FULL data to match the frequency model's test set
train_idx, test_idx = train_test_split(
    data.index, test_size=0.20, random_state=42
)
X_test_full = data.loc[test_idx].reset_index(drop=True)

# Training data: only claimants within the train split
train_claimants = data.loc[train_idx]
train_claimants = train_claimants[
    (train_claimants["n_claims"] > 0) & (train_claimants["claim_amount"] > 0)
].reset_index(drop=True)

X_train_sev = train_claimants
y_train_sev = train_claimants["claim_amount"] / train_claimants["n_claims"]
w_train_sev = train_claimants["n_claims"].astype(float)

print(f"Severity training set (claimants in train split): {len(X_train_sev):,}")
print(f"Scoring set (full test split)                   : {len(X_test_full):,}")

## 02. Preprocessing Pipeline

Identical feature treatment to the frequency model. The same `Pipeline + FeatureUnion`
pattern is used — only the model step and target change.

In [ ]:
# feature_1: policyholder DOB → age
# Younger policyholders may have lower severity (smaller/cheaper vehicles),
# while certain age groups may be involved in higher-severity accidents.
age_pipeline = Pipeline([
    ("compute_age", ComputeAgeFromDOB(dob_column="feature_1")),
    ("imputer",     SimpleImputer(strategy="median")),
    ("scaler",      StandardScaler()),
])

# feature_2: vehicle model year → vehicle age
# Newer vehicles have higher repair and replacement costs → higher severity.
# Older vehicles often have lower insured value → lower claim amounts.
vehicle_age_pipeline = Pipeline([
    ("compute_veh_age", ComputeVehicleAge(year_column="feature_2")),
    ("imputer",         SimpleImputer(strategy="median")),
    ("scaler",          StandardScaler()),
])

# feature_3: insured value — primary driver of claim severity
# Higher insured value directly constrains the maximum claim size and
# correlates with repair/replacement costs.
# features 4, 5: geographic coordinates
numerical_pipeline = Pipeline([
    ("selector", FeatureSelectorValues(["feature_3", "feature_4", "feature_5"])),
    ("imputer",  SimpleImputer(strategy="median")),
    ("scaler",   StandardScaler()),
])

# feature_6: geographic region (20 levels)
# Regional severity patterns reflect local repair shop costs, parts availability,
# and legal/medical expense norms.
region_pipeline = Pipeline([
    ("selector", FeatureSelectorValues(["feature_6"])),
    ("imputer",  SimpleImputer(strategy="most_frequent")),
    ("encoder",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# feature_7: vehicle category (10 levels)
vehicle_cat_pipeline = Pipeline([
    ("selector", FeatureSelectorValues(["feature_7"])),
    ("imputer",  SimpleImputer(strategy="most_frequent")),
    ("encoder",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# feature_8: binary categorical attribute
binary_pipeline = Pipeline([
    ("selector", FeatureSelectorValues(["feature_8"])),
    ("imputer",  SimpleImputer(strategy="most_frequent")),
    ("encoder",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# feature_9: coverage type (5 levels)
# Coverage scope directly determines which losses are eligible for payment,
# affecting observed severity.
coverage_pipeline = Pipeline([
    ("selector", FeatureSelectorValues(["feature_9"])),
    ("imputer",  SimpleImputer(strategy="most_frequent")),
    ("encoder",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# features 10–14: geo-numerical enrichment
geo_pipeline = Pipeline([
    ("selector", FeatureSelectorValues(
        ["feature_10", "feature_11", "feature_12", "feature_13", "feature_14"]
    )),
    ("imputer",  SimpleImputer(strategy="constant", fill_value=0)),
])

preprocessing = FeatureUnion(transformer_list=[
    ("age",             age_pipeline),
    ("vehicle_age",     vehicle_age_pipeline),
    ("numerical",       numerical_pipeline),
    ("region",          region_pipeline),
    ("vehicle_category",vehicle_cat_pipeline),
    ("binary",          binary_pipeline),
    ("coverage",        coverage_pipeline),
    ("geo",             geo_pipeline),
])

print("Preprocessing pipeline assembled.")

## 03. Baseline: GLM with Gamma Regression

The Gamma GLM with log link is the classical actuarial approach to severity
modelling. It assumes the relationship between features and severity is
**log-linear** (multiplicative effects on the cost per claim).

In [ ]:
glm_sev = Pipeline([
    ("preprocessing", preprocessing),
    ("model",         GammaRegressor(alpha=0.1, max_iter=300)),
])

glm_sev.fit(X_train_sev, y_train_sev, model__sample_weight=w_train_sev)
print("GLM (Gamma) — training complete.")

In [ ]:
# Evaluate on claimants in the test split only
test_claimants = X_test_full[
    (X_test_full["n_claims"] > 0) & (X_test_full["claim_amount"] > 0)
].reset_index(drop=True)
y_test_sev = test_claimants["claim_amount"] / test_claimants["n_claims"]
w_test_sev = test_claimants["n_claims"].astype(float)

glm_sev_pred_test_claimants = glm_sev.predict(test_claimants)
print("GLM — Mean Gamma Deviance (lower is better)")
print(f"  Test (claimants): {mean_gamma_deviance(y_test_sev, glm_sev_pred_test_claimants, sample_weight=w_test_sev):.5f}")

## 04. Challenger: XGBoost with Gamma Objective

XGBoost with `objective='reg:gamma'` minimises the Gamma deviance with
gradient-boosted trees, capturing non-linear effects and interactions.

In [ ]:
xgb_sev_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("model", XGBRegressor(
        objective="reg:gamma",
        n_jobs=-1,
        random_state=42,
        verbosity=0,
    )),
])

params_sev = {
    "model__n_estimators":     [30, 60, 100],
    "model__learning_rate":    [0.01, 0.10, 0.15, 0.20],
    "model__min_child_weight": [1, 3, 10, 15, 20],
    "model__subsample":        [0.8, 1.0],
    "model__colsample_bytree": [1.0],
    "model__max_depth":        [3, 5, 8, 10],
}

xgb_sev_cv = GridSearchCV(
    estimator=xgb_sev_pipeline,
    param_grid=params_sev,
    scoring="neg_mean_gamma_deviance",
    cv=3,
    verbose=1,
    n_jobs=-1,
)

In [ ]:
xgb_sev_cv.fit(X_train_sev, y_train_sev, model__sample_weight=w_train_sev)

print(f"Best CV Gamma deviance : {-xgb_sev_cv.best_score_:.5f}")
print("\nBest parameters:")
for k, v in xgb_sev_cv.best_params_.items():
    print(f"  {k:35s}: {v}")

In [ ]:
xgb_sev_best = xgb_sev_cv.best_estimator_
xgb_sev_pred_test_claimants = xgb_sev_best.predict(test_claimants)

dev_glm = mean_gamma_deviance(y_test_sev, glm_sev_pred_test_claimants, sample_weight=w_test_sev)
dev_xgb = mean_gamma_deviance(y_test_sev, xgb_sev_pred_test_claimants, sample_weight=w_test_sev)

print("Mean Gamma Deviance — test claimants (lower is better)")
print(f"  GLM (Gamma)                  : {dev_glm:.5f}")
print(f"  XGBoost (reg:gamma)          : {dev_xgb:.5f}")
print(f"  Improvement                  : {(dev_glm - dev_xgb) / dev_glm:.1%}")

In [ ]:
# Actual vs predicted severity scatter — test claimants
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, preds, title in zip(
    axes,
    [glm_sev_pred_test_claimants, xgb_sev_pred_test_claimants],
    ["GLM (Gamma)", "XGBoost (reg:gamma)"],
):
    cap = y_test_sev.quantile(0.98)
    ax.scatter(
        np.minimum(preds, cap),
        np.minimum(y_test_sev, cap),
        alpha=0.15, s=5, rasterized=True,
    )
    ax.plot([0, cap], [0, cap], "r--", linewidth=1, label="Perfect prediction")
    ax.set_xlabel("Predicted severity")
    ax.set_ylabel("Actual severity")
    ax.set_title(title)
    ax.legend(fontsize=8)
plt.suptitle("Actual vs predicted severity — test claimants (capped at p98)", y=1.02)
plt.tight_layout()
plt.show()

## 05. Save Models and Test Predictions

Both severity models predict on the **full test set** (not just claimants),
so the pure premium can be assembled as `freq × sev` for every policy.
Results are merged with `data/freq_predictions.csv` and saved as
`data/all_predictions.csv` for `03_validation.ipynb`.

In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(glm_sev,      "models/glm_severity.pkl")
joblib.dump(xgb_sev_best, "models/xgb_severity.pkl")
print("Models saved → models/glm_severity.pkl, models/xgb_severity.pkl")

# Predict severity on the FULL test set (including zero-claim policies)
# PP = frequency × severity requires severity scored on every policy
glm_sev_pred_full  = glm_sev.predict(X_test_full)
xgb_sev_pred_full  = xgb_sev_best.predict(X_test_full)

# Load frequency predictions and merge
freq_pred = pd.read_csv("data/freq_predictions.csv")

all_predictions = freq_pred.copy()
all_predictions["sev_glm"]        = glm_sev_pred_full
all_predictions["sev_xgb"]        = xgb_sev_pred_full

# Pure premium = annual frequency rate × severity per claim
all_predictions["pp_base"]        = all_predictions["freq_base"]        * glm_sev_pred_full
all_predictions["pp_challenger"]  = all_predictions["freq_challenger"]  * xgb_sev_pred_full

# Actual pure premium = actual claim amount / exposure
all_predictions["actual_pp"] = (
    all_predictions["claim_amount"] / all_predictions["exposure"]
)

all_predictions.to_csv("data/all_predictions.csv", index=False)
print(f"All predictions saved → data/all_predictions.csv ({len(all_predictions):,} rows)")
all_predictions.head(3)